# Example-05: Uncoupled twiss parameters

In [1]:
# Import

import numpy
import pandas
import torch
import yaml

import sys
sys.path.append('..')

from harmonica.util import mod
from harmonica.statistics import mean, variance
from harmonica.statistics import weighted_mean, weighted_variance
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter
from harmonica.decomposition import Decomposition
from harmonica.model import Model
from harmonica.table import Table

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())

True


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# In this example twiss parameters are computed for uncoupled model
# Uncoupled model is generated from twiss data

In [4]:
# Load uncoupled model

model = Model(path='../config_uncoupled.yaml', model='uncoupled', dtype=dtype, device=device)

In [5]:
# Generate one-turn matrix at the 'HEAD' location

matrix = model.matrix(0, model.size)
print(matrix)

from harmonica.parameterization import is_symplectic
print(is_symplectic(matrix))

tensor([[-8.174741941785e-01, -1.715086192481e+00, 0.000000000000e+00, 0.000000000000e+00],
        [4.490851430614e-02, -1.129060750494e+00, 0.000000000000e+00, 0.000000000000e+00],
        [0.000000000000e+00, 0.000000000000e+00, -1.189482037211e+00, -7.354804784961e+00],
        [0.000000000000e+00, 0.000000000000e+00, 4.179161971876e-02, -5.822961370188e-01]],
       dtype=torch.float64)
True


In [6]:
# Wolski twiss parameters for a given input matrix can be computed with twiss_compute
# This method returns tuple of tunes, normalization matrix and Wolski twiss matrices for each plane
# Note, input matrix can have arbitrary even dimension

from harmonica.parameterization import twiss_compute
tunes, normal, wolski = twiss_compute(matrix)

In [7]:
# Compare tunes with model tunes
# Note, only fractional part can be computed from the one-turn matrix

print(torch.stack([model.nux - 8.0, model.nuy - 7.0]))
print(tunes)

tensor([5.368830987374e-01, 5.767746333258e-01], dtype=torch.float64)
tensor([5.368830987374e-01, 5.767746333258e-01], dtype=torch.float64)


In [8]:
# Check normalization matrix

# Standart gauge is used to condition normalization matrix
# M = N R N^-1, R -- rotation matrix
# N[0, 0] > 0, N[2, 2] > 0, ...
# N[0, 1] = N[2, 3] = ... = 0

print(normal)

mux, muy = 2.0*numpy.pi*tunes
rotation = torch.tensor([
    [mux.cos(), +mux.sin(), 0, 0],
    [-mux.sin(), mux.cos(), 0, 0],
    [0, 0, muy.cos(), +muy.sin()],
    [0, 0, -muy.sin(), muy.cos()]
])

print(torch.allclose(matrix, normal @ rotation @ normal.inverse()))

tensor([[2.732665537666e+00, 0.000000000000e+00, 0.000000000000e+00, 0.000000000000e+00],
        [2.482271293933e-01, 3.659430640949e-01, 0.000000000000e+00, 0.000000000000e+00],
        [0.000000000000e+00, 0.000000000000e+00, 3.981756917048e+00, 0.000000000000e+00],
        [0.000000000000e+00, 0.000000000000e+00, -1.643596756619e-01, 2.511454166673e-01]],
       dtype=torch.float64)
True


In [9]:
# Compute Wolski twiss matrices

from harmonica.parameterization import id_symplectic

print(wolski)

wx, wy = wolski
s = id_symplectic(2)

print(torch.allclose(matrix, wx @ s * mux.sin() - (wx @ s) @ (wx @ s) * mux.cos() + wy @ s * muy.sin() - (wy @ s) @ (wy @ s) * muy.cos()))

tensor([[[7.467460940747e+00, 6.783217220068e-01, 0.000000000000e+00, 0.000000000000e+00],
         [6.783217220068e-01, 1.955310339260e-01, 0.000000000000e+00, 0.000000000000e+00],
         [0.000000000000e+00, 0.000000000000e+00, 0.000000000000e+00, 0.000000000000e+00],
         [0.000000000000e+00, 0.000000000000e+00, 0.000000000000e+00, 0.000000000000e+00]],

        [[0.000000000000e+00, 0.000000000000e+00, 0.000000000000e+00, 0.000000000000e+00],
         [0.000000000000e+00, 0.000000000000e+00, 0.000000000000e+00, 0.000000000000e+00],
         [0.000000000000e+00, 0.000000000000e+00, 1.585438814646e+01, -6.544402754504e-01],
         [0.000000000000e+00, 0.000000000000e+00, -6.544402754504e-01, 9.008812329666e-02]]],
       dtype=torch.float64)
True


In [10]:
# Convert Wolski to CS and compare with model values

from harmonica.parameterization import wolski_to_cs

print(wolski_to_cs(wolski))
print(torch.stack([model.ax[0], model.bx[0], model.ay[0], model.by[0]]))

tensor([-6.783217220068e-01, 7.467460940747e+00, 6.544402754504e-01, 1.585438814646e+01],
       dtype=torch.float64)
tensor([-6.783217220068e-01, 7.467460940747e+00, 6.544402754504e-01, 1.585438814646e+01],
       dtype=torch.float64)


In [11]:
# Wolski twiss matrices can be propagated using transport matrices between locations

# Define transport matrices from 'HEAD' to each location
# Note, the last matrix is 'HEAD' to 'TAIL'

index = torch.arange(model.size, dtype=torch.int64, device=device)
transport = model.matrix(0*index, index)

# Propagate twiss

from harmonica.parameterization import twiss_propagate
table = twiss_propagate(wolski, transport)
print(table.shape)

# Compare with model

result = [torch.allclose(wolski_to_cs(w), torch.stack([model.ax[i], model.bx[i], model.ay[i], model.by[i]])) for i, w in enumerate(table)]
result.count(True)

torch.Size([59, 2, 4, 4])


59

In [12]:
# Phase advance between locations can be computed using twiss_phase_advance
# For given normalization matrix at the initial location and given transport matrices to other locations
# This function returns phase advance (mod two pi) and normalization matrix at the other locations

from harmonica.parameterization import twiss_phase_advance

advance, _ = twiss_phase_advance(normal, model.matrix('STP2', 'STP4').unsqueeze(0))
print(advance.squeeze())

print(torch.stack([model.fx[model.get_index('STP4')] - model.fx[model.get_index('STP2')], model.fy[model.get_index('STP4')] - model.fy[model.get_index('STP2')]]))

tensor([8.375209565894e-01, 4.070908562792e-01], dtype=torch.float64)
tensor([8.375209565894e-01, 4.070908562792e-01], dtype=torch.float64)


In [13]:
# Compute phase advance & normalization matrix between all adjacent locations and compare with model values

tunes, normal, wolski = twiss_compute(matrix)

total = torch.zeros_like(tunes)

for i in range(model.size):
    advance, normal = twiss_phase_advance(normal, model.matrix(i, i + 1).unsqueeze(0))
    advance = advance.squeeze()
    normal = normal.squeeze()
    total += mod(advance + 1.0E-12, 2.0*numpy.pi) - 1.0E-12
    j = int(mod(i + 1, model.size))
    ax, bx = model.ax[j], model.bx[j]
    ay, by = model.ay[j], model.by[j]
    nx = torch.tensor([[bx.sqrt(), 0], [-ax/bx.sqrt(), 1/bx.sqrt()]])
    ny = torch.tensor([[by.sqrt(), 0], [-ay/by.sqrt(), 1/by.sqrt()]])
    is_advance = torch.allclose(advance, mod(torch.stack([model.phase_x[i], model.phase_y[i]]), 2*numpy.pi))
    is_normal = torch.allclose(normal, torch.block_diag(nx, ny))
    print(f'{model.get_name(i):>6}{model.get_name(i + 1):>6} {is_advance} {is_normal}')

print()
print(total)
print(torch.stack([model.mux, model.muy]))

  HEAD  STP2 True True
  STP2  IV4P True True
  IV4P  STP4 True True
  STP4  SRP1 True True
  SRP1  SRP2 True True
  SRP2  SRP3 True True
  SRP3  SRP4 True True
  SRP4  SRP5 True True
  SRP5  SRP6 True True
  SRP6  SRP7 True True
  SRP7  SRP8 True True
  SRP8  SRP9 True True
  SRP9  SIP1 True True
  SIP1  SIP2 True True
  SIP2 SRP10 True True
 SRP10 SRP11 True True
 SRP11 SRP12 True True
 SRP12 SRP13 True True
 SRP13 SRP14 True True
 SRP14 SRP15 True True
 SRP15 SRP16 True True
 SRP16 SRP17 True True
 SRP17  SEP5 True True
  SEP5  SEP4 True True
  SEP4  SEP3 True True
  SEP3  SEP1 True True
  SEP1  SEP0 True True
  SEP0    IP True True
    IP  NEP0 True True
  NEP0  NEP1 True True
  NEP1  NEP3 True True
  NEP3  NEP4 True True
  NEP4  NEP5 True True
  NEP5 NRP17 True True
 NRP17 NRP16 True True
 NRP16 NRP15 True True
 NRP15 NRP14 True True
 NRP14 NRP13 True True
 NRP13 NRP12 True True
 NRP12 NRP11 True True
 NRP11 NRP10 True True
 NRP10  NIP3 True True
  NIP3  NIP1 True True
  NIP1  NRP

In [14]:
# Since twiss method can handle arbitrary even dimension matrices
# One can construct a block matrix from one-turn matrices at all locations and compute corresponding twiss parameters

index = torch.arange(model.size, dtype=torch.int64, device=device)
matrix = torch.block_diag(*model.matrix(index, index + model.size))
print(matrix.shape)

_, normal, table = twiss_compute(matrix)
print(table.shape)

# Exctact beta values and compare with model

print()
print(torch.allclose(model.bx, torch.stack([table[2*i + 0, 4*i + 0, 4*i + 0] for i in range(model.size)])))
print(torch.allclose(model.by, torch.stack([table[2*i + 1, 4*i + 2, 4*i + 2] for i in range(model.size)])))

print()
print(torch.allclose(model.bx, torch.stack([normal[4*i + 0, 4*i + 0]**2 for i in range(model.size)])))
print(torch.allclose(model.by, torch.stack([normal[4*i + 2, 4*i + 2]**2 for i in range(model.size)])))

torch.Size([236, 236])
torch.Size([118, 236, 236])

True
True

True
True
